In [2]:
!python -c "import sys; print(sys.executable)"

e:\2-git_repositories\12-autonomous-mobile-robot\robot-third-assignmnet\.venv\Scripts\python.exe


In [2]:
!python -m pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import open3d as o3d
import numpy as np
import copy
import os

def manual_transform_point_cloud(pcd, transformation_matrix):
    """
    Manually applies a 4x4 rigid transformation matrix to an Open3D point cloud using NumPy.
    """
    # 1. Extract Nx3 points to a NumPy array
    points = np.asarray(pcd.points)
    
    # 2. Convert to homogeneous coordinates (Nx4) by appending 1s
    ones = np.ones((points.shape[0], 1))
    points_homogeneous = np.hstack((points, ones))
    
    # 3. Apply the transformation matrix: P_new = T * P_old^T
    transformed_points_homogeneous = transformation_matrix @ points_homogeneous.T
    
    # 4. Transpose back to Nx4 and drop the homogeneous coordinate to get Nx3
    transformed_points = transformed_points_homogeneous.T[:, :3]
    
    # 5. Assign the new points to a deep copy of the point cloud
    pcd_transformed = copy.deepcopy(pcd)
    pcd_transformed.points = o3d.utility.Vector3dVector(transformed_points)
    
    # 6. Manually rotate normals if they exist
    if pcd.has_normals():
        normals = np.asarray(pcd.normals)
        rotation_matrix = transformation_matrix[:3, :3] # Extract 3x3 rotation
        transformed_normals = normals @ rotation_matrix.T
        pcd_transformed.normals = o3d.utility.Vector3dVector(transformed_normals)
        
    return pcd_transformed

def manual_merge_point_clouds(pcd1, pcd2):
    """
    Manually concatenates two point clouds by stacking their NumPy arrays.
    """
    points1 = np.asarray(pcd1.points)
    points2 = np.asarray(pcd2.points)
    merged_points = np.vstack((points1, points2))
    
    merged_cloud = o3d.geometry.PointCloud()
    merged_cloud.points = o3d.utility.Vector3dVector(merged_points)
    return merged_cloud

def draw_registration_result(source, target, transformation, name_of_file="aligned_cloud"):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    
    source_temp = manual_transform_point_cloud(source_temp, transformation)
    
    o3d.visualization.draw_geometries([source_temp, target_temp])
    
    merged_cloud = manual_merge_point_clouds(source_temp, target_temp)
    o3d.io.write_point_cloud(f"{name_of_file}.pcd", merged_cloud)

# 1. Define paths to your specific files
cloud_1 = "./sampled_clouds_for_icp/001_cloud_1683269379_452476000_icp_sampled.pcd"
cloud_2 = "./sampled_clouds_for_icp/002_cloud_1683269381_551481000_icp_sampled.pcd"

# 2. Load the point clouds
source = o3d.io.read_point_cloud(cloud_1)
target = o3d.io.read_point_cloud(cloud_2)

# 3. Preprocessing (Voxel Downsampling)
voxel_size = 0.5 
source_down = source.voxel_down_sample(voxel_size)
target_down = target.voxel_down_sample(voxel_size)

# Estimate normals (Required for Point-to-Plane ICP)
source_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
target_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))

# 4. Initialization
initial_guess = np.identity(4)

print("Visualizing Before Alignment (Source=Yellow, Target=Blue)...")
draw_registration_result(source_down, target_down, initial_guess, name_of_file="before_alignment")

Visualizing Before Alignment (Source=Yellow, Target=Blue)...


In [4]:
# 5. Run ICP
threshold = 1.0 
print("Running Point-to-Plane ICP...")
reg_p2l = o3d.pipelines.registration.registration_icp(
    source_down, target_down, threshold, initial_guess,
    o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

# 6. Report Metrics
print("Estimated Rigid Transformation Matrix:")
print(reg_p2l.transformation)
print(f"Fitness Score: {reg_p2l.fitness}")
print(f"Inlier RMSE: {reg_p2l.inlier_rmse}")

print("Visualizing After Alignment...")
draw_registration_result(source_down, target_down, reg_p2l.transformation, name_of_file="after_alignment")

Running Point-to-Plane ICP...
Estimated Rigid Transformation Matrix:
[[ 0.9838156  -0.00206147  0.17917202 -0.57363832]
 [ 0.0090297   0.99923372 -0.03808448 -0.16112656]
 [-0.17895622  0.03908598  0.98308034 -0.04614291]
 [ 0.          0.          0.          1.        ]]
Fitness Score: 0.3480653017046992
Inlier RMSE: 0.6299117385458369
Visualizing After Alignment...


# use grid search for finding best result

In [5]:
import open3d as o3d
import numpy as np
import copy

# (Assume 'source' and 'target' are already loaded here)

# 1. Setup tracking variables for the "Best" result
best_fitness = -1.0
best_rmse = float('inf')
best_transformation = None
best_params = {}

print("Starting grid search for optimal ICP parameters...")

# 2. Run the Grid Search
for voxel_size in [0.5, 0.3, 0.1]:
    for threshold in [1.0, 2.0, 3.0]:
        for max_iter in [500, 1000, 2000]:
            for type_of_icp in ["point_to_point", "point_to_plane"]:
                print(f"Running ICP with voxel_size={voxel_size}, threshold={threshold}, max_iter={max_iter}, type={type_of_icp}")
                # Downsample
                source_down = source.voxel_down_sample(voxel_size)
                target_down = target.voxel_down_sample(voxel_size)

                # Estimate normals (Required for Point-to-Plane, but harmless to run for both)
                source_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
                target_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))

                # Setup the correct estimation method based on the loop variable
                if type_of_icp == "point_to_plane":
                    estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPlane()
                else:
                    estimation_method = o3d.pipelines.registration.TransformationEstimationPointToPoint()

                initial_guess = np.identity(4)

                # Run ICP using the dynamic loop variables
                reg_result = o3d.pipelines.registration.registration_icp(
                    source_down, target_down, threshold, initial_guess,
                    estimation_method,
                    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=max_iter)
                )

                # 3. Evaluate if this run is better than our previous best
                # Primary goal: Maximize Fitness. Secondary goal: Minimize RMSE
                print(f"Fitness: {reg_result.fitness}, Inlier RMSE: {reg_result.inlier_rmse}")
                is_better_fitness = reg_result.fitness > best_fitness
                is_tied_but_better_rmse = (reg_result.fitness == best_fitness) and (reg_result.inlier_rmse < best_rmse)

                if is_better_fitness or is_tied_but_better_rmse:
                    best_fitness = reg_result.fitness
                    best_rmse = reg_result.inlier_rmse
                    best_transformation = reg_result.transformation
                    best_params = {
                        "voxel_size": voxel_size,
                        "threshold": threshold,
                        "max_iter": max_iter,
                        "type_of_icp": type_of_icp
                    }

# 4. Output the final winning results
print("\n=== BEST PARAMETERS FOUND ===")
print(f"Parameters: {best_params}")
print(f"Best Fitness: {best_fitness}")
print(f"Best Inlier RMSE: {best_rmse}")
print("Best Transformation Matrix:\n", best_transformation)

# 5. Visualize the winning alignment
print("\nVisualizing best alignment...")
source_temp = copy.deepcopy(source)
target_temp = copy.deepcopy(target)

source_temp.paint_uniform_color([1, 0.706, 0])      # Yellow
target_temp.paint_uniform_color([0, 0.651, 0.929])  # Blue

# Apply the best transformation to the source
source_temp.transform(best_transformation)

# Draw the result
o3d.visualization.draw_geometries([source_temp, target_temp])

Starting grid search for optimal ICP parameters...
Running ICP with voxel_size=0.5, threshold=1.0, max_iter=500, type=point_to_point
Fitness: 0.34495354920176785, Inlier RMSE: 0.6255375621573913
Running ICP with voxel_size=0.5, threshold=1.0, max_iter=500, type=point_to_plane
Fitness: 0.3480653017046992, Inlier RMSE: 0.629911738545837
Running ICP with voxel_size=0.5, threshold=1.0, max_iter=1000, type=point_to_point
Fitness: 0.34495354920176785, Inlier RMSE: 0.6255375621573901
Running ICP with voxel_size=0.5, threshold=1.0, max_iter=1000, type=point_to_plane
Fitness: 0.3480653017046992, Inlier RMSE: 0.6299117385458368
Running ICP with voxel_size=0.5, threshold=1.0, max_iter=2000, type=point_to_point
Fitness: 0.34495354920176785, Inlier RMSE: 0.6255375621573895
Running ICP with voxel_size=0.5, threshold=1.0, max_iter=2000, type=point_to_plane
Fitness: 0.3480653017046992, Inlier RMSE: 0.6299117385458369
Running ICP with voxel_size=0.5, threshold=2.0, max_iter=500, type=point_to_point
Fit

# Coarse-to-Fine ICP Code

In [6]:
import open3d as o3d
import numpy as np
import copy

print("--- STAGE 1: COARSE ALIGNMENT ---")
# 1. Setup Coarse Parameters (Fast & Wide)
voxel_size_coarse = 0.5  # Large voxels for speed
threshold_coarse = 3.0   # Large threshold to catch massive overlaps (from your grid search)

source_down_coarse = source.voxel_down_sample(voxel_size_coarse)
target_down_coarse = target.voxel_down_sample(voxel_size_coarse)

initial_guess = np.identity(4)

# Run Coarse ICP (Point-to-Point is usually safer for rough alignment)
reg_coarse = o3d.pipelines.registration.registration_icp(
    source_down_coarse, target_down_coarse, threshold_coarse, initial_guess,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

print(f"Coarse Fitness: {reg_coarse.fitness:.4f}")
print(f"Coarse Inlier RMSE: {reg_coarse.inlier_rmse:.4f}")


print("\n--- STAGE 2: FINE ALIGNMENT ---")
# 2. Setup Fine Parameters (Detailed & Strict)
voxel_size_fine = 0.1   # Smaller voxels for high structural detail
threshold_fine = 2.0    # Extremely tight threshold (e.g., 20 cm)

source_down_fine = source.voxel_down_sample(voxel_size_fine)
target_down_fine = target.voxel_down_sample(voxel_size_fine)

# Estimate normals for Point-to-Plane (Crucial for tight architectural snapping)
source_down_fine.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_fine * 2, max_nn=30))
target_down_fine.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_fine * 2, max_nn=30))

# The secret to Coarse-to-Fine: Pass the matrix from Stage 1 into Stage 2
coarse_transformation = reg_coarse.transformation

# Run Fine ICP (Point-to-Plane)
reg_fine = o3d.pipelines.registration.registration_icp(
    source_down_fine, target_down_fine, threshold_fine, coarse_transformation,
    o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

print(f"Fine Fitness: {reg_fine.fitness:.4f}")
print(f"Fine Inlier RMSE: {reg_fine.inlier_rmse:.4f}")
print("\nFinal Optimal Transformation Matrix:")
print(reg_fine.transformation)


print("\nVisualizing Final Output...")
# We visualize using the fine downsampled clouds for a clearer picture
draw_registration_result(source_down_fine, target_down_fine, reg_fine.transformation)

--- STAGE 1: COARSE ALIGNMENT ---
Coarse Fitness: 0.8617
Coarse Inlier RMSE: 1.5171

--- STAGE 2: FINE ALIGNMENT ---
Fine Fitness: 0.7017
Fine Inlier RMSE: 1.0947

Final Optimal Transformation Matrix:
[[ 0.92721148  0.33028047  0.17661734 -0.42498573]
 [-0.32892201  0.94360104 -0.03778073 -2.5492674 ]
 [-0.17913454 -0.0230626   0.98355423 -0.45524708]
 [ 0.          0.          0.          1.        ]]

Visualizing Final Output...


# change the treshold to 1.5

In [7]:
import open3d as o3d
import numpy as np
import copy


print("--- STAGE 1: COARSE ALIGNMENT ---")
# 1. Setup Coarse Parameters (Fast & Wide)
voxel_size_coarse = 0.5  # Large voxels for speed
threshold_coarse = 3.0   # Large threshold to catch massive overlaps (from your grid search)

source_down_coarse = source.voxel_down_sample(voxel_size_coarse)
target_down_coarse = target.voxel_down_sample(voxel_size_coarse)

initial_guess = np.identity(4)

# Run Coarse ICP (Point-to-Point is usually safer for rough alignment)
reg_coarse = o3d.pipelines.registration.registration_icp(
    source_down_coarse, target_down_coarse, threshold_coarse, initial_guess,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

print(f"Coarse Fitness: {reg_coarse.fitness:.4f}")
print(f"Coarse Inlier RMSE: {reg_coarse.inlier_rmse:.4f}")


print("\n--- STAGE 2: FINE ALIGNMENT ---")
# 2. Setup Fine Parameters (Detailed & Strict)
voxel_size_fine = 0.1   # Smaller voxels for high structural detail
threshold_fine = 1.5    # Extremely tight threshold (e.g., 20 cm)

source_down_fine = source.voxel_down_sample(voxel_size_fine)
target_down_fine = target.voxel_down_sample(voxel_size_fine)

# Estimate normals for Point-to-Plane (Crucial for tight architectural snapping)
source_down_fine.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_fine * 2, max_nn=30))
target_down_fine.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_fine * 2, max_nn=30))

# The secret to Coarse-to-Fine: Pass the matrix from Stage 1 into Stage 2
coarse_transformation = reg_coarse.transformation

# Run Fine ICP (Point-to-Plane)
reg_fine = o3d.pipelines.registration.registration_icp(
    source_down_fine, target_down_fine, threshold_fine, coarse_transformation,
    o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
)

print(f"Fine Fitness: {reg_fine.fitness:.4f}")
print(f"Fine Inlier RMSE: {reg_fine.inlier_rmse:.4f}")
print("\nFinal Optimal Transformation Matrix:")
print(reg_fine.transformation)


print("\nVisualizing Final Output...")
# We visualize using the fine downsampled clouds for a clearer picture
draw_registration_result(source_down_fine, target_down_fine, reg_fine.transformation)

--- STAGE 1: COARSE ALIGNMENT ---
Coarse Fitness: 0.8617
Coarse Inlier RMSE: 1.5171

--- STAGE 2: FINE ALIGNMENT ---
Fine Fitness: 0.5693
Fine Inlier RMSE: 0.8595

Final Optimal Transformation Matrix:
[[ 0.98320023  0.04252532  0.17750745  1.57369085]
 [-0.03769774  0.99882415 -0.03048264 -1.16666553]
 [-0.17859501  0.02327891  0.98364725 -0.53407007]
 [ 0.          0.          0.          1.        ]]

Visualizing Final Output...


# task 2

In [1]:
import open3d as o3d
import numpy as np
import glob
import copy
import os

def manual_transform_point_cloud(pcd, transformation_matrix):
    """
    اعمال دستی ماتریس تبدیل ۴در۴ روی ابر نقطه با استفاده از محاسبات ماتریسی NumPy.
    """
    points = np.asarray(pcd.points)
    
    ones = np.ones((points.shape[0], 1))
    points_homogeneous = np.hstack((points, ones))
    
    transformed_points_homogeneous = transformation_matrix @ points_homogeneous.T
    
    transformed_points = transformed_points_homogeneous.T[:, :3]
    
    pcd_transformed = copy.deepcopy(pcd)
    pcd_transformed.points = o3d.utility.Vector3dVector(transformed_points)
    
    if pcd.has_normals():
        normals = np.asarray(pcd.normals)
        rotation_matrix = transformation_matrix[:3, :3] 
        transformed_normals = normals @ rotation_matrix.T
        pcd_transformed.normals = o3d.utility.Vector3dVector(transformed_normals)
        
    return pcd_transformed

def manual_merge_point_clouds(pcd1, pcd2):
    
    points1 = np.asarray(pcd1.points)
    points2 = np.asarray(pcd2.points)
    
    if len(points1) == 0:
        merged_points = points2
    elif len(points2) == 0:
        merged_points = points1
    else:
        merged_points = np.vstack((points1, points2))
    
    merged_cloud = o3d.geometry.PointCloud()
    merged_cloud.points = o3d.utility.Vector3dVector(merged_points)
    return merged_cloud
# -------------------------------------------------------------

pcd_files = sorted(glob.glob("*.pcd"))
if len(pcd_files) == 0:
    raise FileNotFoundError("هیچ فایل .pcd در پوشه جاری یافت نشد.")

print(f"Found {len(pcd_files)} point clouds. Starting consecutive mapping...")

voxel_size_coarse = 0.5
threshold_coarse = 3.0
voxel_size_fine = 0.1
threshold_fine = 1.5
max_iter = 500

trajectory_points = []

target = o3d.io.read_point_cloud(pcd_files[0])
target_down_fine = target.voxel_down_sample(voxel_size_fine)
target_down_fine.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_fine * 2, max_nn=30))

global_transform = np.identity(4)
trajectory_points.append(global_transform[:3, 3]) 

global_map = copy.deepcopy(target_down_fine)

for i in range(1, len(pcd_files)):
    print(f"Aligning Scan {i:02d} to {i-1:02d}...")
    
    source = o3d.io.read_point_cloud(pcd_files[i])
    
    source_down_coarse = source.voxel_down_sample(voxel_size_coarse)
    target_down_coarse = target.voxel_down_sample(voxel_size_coarse)
    
    reg_coarse = o3d.pipelines.registration.registration_icp(
        source_down_coarse, target_down_coarse, threshold_coarse, np.identity(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=max_iter)
    )
    
    source_down_fine = source.voxel_down_sample(voxel_size_fine)
    source_down_fine.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_fine * 2, max_nn=30))
    
    reg_fine = o3d.pipelines.registration.registration_icp(
        source_down_fine, target_down_fine, threshold_fine, reg_coarse.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=max_iter)
    )
    
    relative_transform = reg_fine.transformation
    
    global_transform = np.dot(global_transform, relative_transform)
    
    trajectory_points.append(global_transform[:3, 3])
    
    source_global = manual_transform_point_cloud(source_down_fine, global_transform)
    
    global_map = manual_merge_point_clouds(global_map, source_global)
    
    if i % 5 == 0:
        global_map = global_map.voxel_down_sample(voxel_size_fine)
        
    target = source
    target_down_fine = source_down_fine

lines = [[i, i+1] for i in range(len(trajectory_points)-1)]
colors = [[1, 0, 0] for i in range(len(lines))] 
trajectory_lineset = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(trajectory_points),
    lines=o3d.utility.Vector2iVector(lines)
)
trajectory_lineset.colors = o3d.utility.Vector3dVector(colors)

print("\nMapping Complete! Visualizing Global Map and Trajectory...")

global_map_final = global_map.voxel_down_sample(voxel_size_fine)

o3d.io.write_point_cloud("task2_accumulated_map.pcd", global_map_final)

o3d.visualization.draw_geometries([global_map_final, trajectory_lineset])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Found 3 point clouds. Starting consecutive mapping...
Aligning Scan 01 to 00...
Aligning Scan 02 to 01...

Mapping Complete! Visualizing Global Map and Trajectory...
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: The handle is invalid. 
